# 02 - Model Training (Decision Tree, KNN, SVM)
**DS 301 Midterm Project - Predicting Machine Failure Using Machine Learning**

**Role:** Student 2 - Model Training

**Paper:** *Predictive Maintenance of Armoured Vehicles using Machine Learning Approaches* (2023)

**Dataset:** AI4I 2020 Predictive Maintenance Dataset

This notebook trains three models with the settings from **Table II** of the paper and saves their predictions for Student 3 (Evaluation).

It does **not** repeat EDA or cleaning. It only uses the files made by `preprocess.py` in **paper mode** (70:30 split, original 6 features).

Before running, create the data files once from the repository root:

```
python preprocess.py --mode paper
```

## 1. Setup
We import only the libraries we need. We also import `build_pipeline()` from our teammate's `preprocess.py`, so we use the same scaling code as the rest of the team.

In [1]:
# Purpose: Import the required libraries and set the working directory to the project root.
# Since this notebook is in notebooks/, move up one level to the project root
# to access the data/ directory and preprocess.py.
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())

import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

from preprocess import build_pipeline   # Reuse the teammate's pipeline builder (scaler + model).

RANDOM_STATE = 42
print("Working folder:", os.getcwd())

Working folder: <project root>


## 2. Load the train / test data
The train/test split was already done in `preprocess.py` (70% train, 30% test, stratified, `random_state=42`).

We **do not split again**. Every model in this notebook uses the same `X_test` and `y_test`.

In [2]:
# Purpose: Load the train/test files generated by preprocessing (do not split again).
DATA_DIR = "data/processed/paper"

X_train = pd.read_csv(f"{DATA_DIR}/X_train.csv")
X_test = pd.read_csv(f"{DATA_DIR}/X_test.csv")
y_train = pd.read_csv(f"{DATA_DIR}/y_train.csv")["machine_failure"]
y_test = pd.read_csv(f"{DATA_DIR}/y_test.csv")["machine_failure"]

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("Failures in train:", y_train.sum(), "/", len(y_train))
print("Failures in test :", y_test.sum(), "/", len(y_test))
X_train.head()

X_train: (7000, 6)  X_test: (3000, 6)
Failures in train: 237 / 7000
Failures in test : 102 / 3000


,type,air_temperature,process_temperature,rotational_speed,torque,tool_wear
0,2,297.8,307.4,1902,24.3,129
1,1,303.7,312.3,1349,51.0,105
2,1,297.2,307.9,1493,38.4,146
3,2,303.6,312.3,1630,32.4,223
4,0,300.9,311.9,2140,16.5,43


## 3. Verify the paper's `Type` encoding
The paper encodes `Type` as **H = 0, L = 1, M = 2**.

Our `preprocess.py` already applies this mapping in **paper mode** when it creates `X_train.csv` and `X_test.csv`.

Therefore, **no additional re-encoding is needed** in this notebook. We use the preprocessed values directly for model training and evaluation.

In [3]:
# Purpose: Verify the Type encoding already applied by preprocess.py.
# Paper mode uses H=0, L=1, M=2. No additional encoding is required.
print(X_train["type"].value_counts().sort_index())

type
0     705
1    4217
2    2078
Name: count, dtype: int64


## 4. Scaling and data leakage
The paper applies **Min-Max Normalization** to the 5 sensor features and leaves `Type` unscaled.

`build_pipeline(model, mode="paper")` does exactly this. It puts `MinMaxScaler` and the model into one scikit-learn `Pipeline`:

- `pipeline.fit(X_train, y_train)` -> the scaler learns min/max from **training data only**
- `pipeline.predict(X_test)` -> the test data is only **transformed** with the training min/max

So the test data never influences the scaler. The saved CSV files are unscaled, which is why we scale here.

## 5. Decision Tree
Paper Table II: `min_samples_split = 2`

We add `random_state=42` only so that we get the same tree every time we run the notebook.

A Decision Tree does not need scaling, but we use the same pipeline for all models to keep the same preprocessing as the paper.

In [4]:
# Purpose: Fit the Decision Tree model and predict labels for the test data.
dt_model = build_pipeline(
    DecisionTreeClassifier(min_samples_split=2, random_state=RANDOM_STATE),
    mode="paper",
)
dt_model.fit(X_train, y_train)
y_pred_decision_tree = dt_model.predict(X_test)

print("Decision Tree trained.")
print("Tree depth:", dt_model.named_steps["model"].get_depth())
print("Predicted failures in test:", y_pred_decision_tree.sum(), "of", len(y_pred_decision_tree))

Decision Tree trained.
Tree depth: 19
Predicted failures in test: 119 of 3000


## 6. K-Nearest Neighbors (KNN)
Paper Table II: `metric = 'minkowski'`, `leaf_size = 20`

The paper does not give `n_neighbors`, so we use the scikit-learn default **K = 5**.

KNN uses distances, so scaling is important. `MinMaxScaler` is applied inside the pipeline.

In [5]:
# Purpose: Fit the KNN model (K=5) and predict labels for the test data.
knn_model = build_pipeline(
    KNeighborsClassifier(n_neighbors=5, metric="minkowski", leaf_size=20),
    mode="paper",
)
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)

print("KNN trained (K=5).")
print("Predicted failures in test:", y_pred_knn.sum(), "of", len(y_pred_knn))

KNN trained (K=5).
Predicted failures in test: 36 of 3000


### 6.1 Compare K = 3, 5, 7, 9 with 5-fold Cross-Validation
We check if K = 5 is a reasonable choice.

- Cross-validation uses the **training data only**. The test data is not touched.
- Because the scaler is inside the pipeline, it is re-fitted in each fold (no leakage between folds).
- We use the **F1 score** because only about 3.4% of samples are failures, so accuracy alone can look good even for a weak model.

This is only a comparison. The main KNN model above stays at K = 5.

In [6]:
# Purpose: Compare K values using 5-fold cross-validation on training data only.
cv_rows = []
for k in [3, 5, 7, 9]:
    pipe = build_pipeline(
        KNeighborsClassifier(n_neighbors=k, metric="minkowski", leaf_size=20),
        mode="paper",
    )
    f1_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="f1")
    acc_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")
    cv_rows.append({
        "K": k,
        "mean_cv_f1": f1_scores.mean(),
        "std_cv_f1": f1_scores.std(),
        "mean_cv_accuracy": acc_scores.mean(),
    })

knn_cv_results = pd.DataFrame(cv_rows).round(4)
knn_cv_results

,K,mean_cv_f1,std_cv_f1,mean_cv_accuracy
0,3,0.4559,0.0666,0.9734
1,5,0.3798,0.0749,0.9730
2,7,0.3657,0.1111,0.9731
3,9,0.3087,0.1084,0.9716


## 7. SVM (optional third model)
Paper Table II: `kernel = 'rbf'`, `gamma = 'scale'`

**Note:** In Table II the row name is *"SVM - Linear Kernel"*, but the listed setting is `kernel: 'rbf'`. The name and the setting do not agree. We follow the **listed hyperparameters** (RBF kernel) and report this as an unclear point in the paper.

SVM also uses distances, so `MinMaxScaler` is applied inside the pipeline.

In [7]:
# Purpose: Fit the SVM model (RBF kernel) and predict labels for the test data.
# In the paper, Table II labels the row "SVM - Linear Kernel" but specifies kernel='rbf'.
# We follow the listed hyperparameters (rbf, gamma='scale').
svm_model = build_pipeline(
    SVC(kernel="rbf", gamma="scale"),
    mode="paper",
)
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)

print("SVM trained (RBF kernel).")
print("Predicted failures in test:", y_pred_svm.sum(), "of", len(y_pred_svm))

SVM trained (RBF kernel).
Predicted failures in test: 0 of 3000


## 8. Save predictions for Student 3 (Evaluation)
All three models predicted on the **same** `X_test`. We save the true labels and each model's predictions in one CSV file, one row per test sample (same row order as `X_test.csv` and `y_test.csv`).

Student 3 will use this file to calculate Accuracy, Precision, Recall, F1 and the Confusion Matrix.

In [8]:
# Purpose: Save the true labels (y_test) and predictions from each model in one CSV.
predictions = pd.DataFrame({
    "y_test": y_test.values,
    "y_pred_decision_tree": y_pred_decision_tree,
    "y_pred_knn": y_pred_knn,
    "y_pred_svm": y_pred_svm,
})

os.makedirs("reports", exist_ok=True)
predictions.to_csv("reports/predictions_paper.csv", index=False)
knn_cv_results.to_csv("reports/knn_cv_results.csv", index=False)

print("Saved: reports/predictions_paper.csv", predictions.shape)
predictions.head()

Saved: reports/predictions_paper.csv (3000, 4)


,y_test,y_pred_decision_tree,y_pred_knn,y_pred_svm
0,0,0,0,0
1,0,0,0,0
2,0,0,0,0
3,0,0,0,0
4,0,0,0,0


In [9]:
# Purpose: Verify equal test prediction counts across models and check for missing values.
print("Rows:", len(predictions), "(should be", len(y_test), ")")
print("Missing values:", predictions.isna().sum().sum())
print()
print("How many 1s (failures) in each column:")
print(predictions.sum())

Rows: 3000 (should be 3000 )
Missing values: 0

How many 1s (failures) in each column:
y_test                  102
y_pred_decision_tree    119
y_pred_knn               36
y_pred_svm                0
dtype: int64


## 9. To add for full paper reproduction: SMOTE (not done in this notebook)
The paper balances its **training set** using SMOTE (failure samples 240 -> 6,760).

This notebook reports the **baseline without SMOTE**. SMOTE is already supported by `build_pipeline()` in `preprocess.py`, and `imbalanced-learn` is included in `requirements.txt`.

To extend the reproduction, retrain the models with SMOTE enabled **inside the pipeline**, so it runs on training data (and training folds) only, never on validation or test samples:

```python
# Example for a future experiment (not executed in the current baseline):
# model = build_pipeline(
#     DecisionTreeClassifier(min_samples_split=2, random_state=42),
#     mode="paper", smote=True,
# )
# model.fit(X_train, y_train)
```

Keep SMOTE-based predictions separate from the current baseline results. Until the extension is implemented, our results should be described as **"paper settings without SMOTE"**.